# PS-1 — TF-IDF vs Word2Vec / GloVe

The brief asks you to "build features with TF-IDF and n-grams as a baseline, then compare against
Word2Vec / GloVe embeddings". Script 03 did the baseline; this does the comparison.

This is the Colab version, because gensim has no prebuilt wheel for Python 3.14 and compiling it
locally needs Visual C++ Build Tools. Colab ships gensim already, and `colab_ps1.csv` is in your
Drive from notebook 05.

**No GPU needed** — this is all CPU work, so leave the runtime as-is.

### What the comparison is actually testing

TF-IDF gives every word its own column, so "refund" and "reimbursement" are unrelated — a ticket
saying one gets no benefit from all the training examples that said the other.

Word2Vec learns a dense vector per word from the company it keeps, so related words land near each
other. The catch: a ticket is a sentence, and to get one vector per ticket you average the word
vectors — which throws away word order entirely. "not working" and "working" average to nearly the
same point.

    TF-IDF     : sparse, no notion of meaning, keeps exact words and bigrams
    embeddings : dense, knows word similarity, blurs the sentence together

Which wins tells you something real about your data, and either answer is reportable.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/support_ticket_project'

# Your best TF-IDF macro-F1 from script 03. Change it if yours differs.
TFIDF_BASELINE = 0.971

import os
assert os.path.exists(f'{DRIVE}/colab_ps1.csv'), 'colab_ps1.csv not found in Drive'
print('data found')

In [ ]:
!pip install -q gensim
import gensim; print('gensim', gensim.__version__)

In [ ]:
import re
import numpy as np
import pandas as pd
from gensim.models import Word2Vec
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
VECTOR_SIZE = 200
W2V_EPOCHS = 10
USE_GLOVE = True        # downloads glove-twitter-200 (~130 MB) - fast on Colab

# Tokenizer, and why it is not just .split():
# "refund", "refund.", "refund?" and "refund!" would be four separate words,
# each trained on a quarter of the data. Strip punctuation or the embedding
# learns punctuation instead of meaning.
TOKEN = re.compile(r"[a-z0-9']+")
def tokenize(t):
    return TOKEN.findall(str(t).lower())

df = pd.read_csv(f'{DRIVE}/colab_ps1.csv').dropna(subset=['clean_text', 'intent'])
print(f'{len(df):,} rows')
print(df['intent'].value_counts().to_string())

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'].values, df['intent'].values,
    test_size=0.2, random_state=RANDOM_STATE, stratify=df['intent'].values)
print(f'\ntrain {len(X_train):,}   test {len(X_test):,}')

In [ ]:
# Trained on TRAINING TEXTS ONLY - training on everything would let test
# vocabulary shape the vectors, which quietly inflates the score.
# sg=1 is skip-gram: slower than CBOW but better on small corpora and rare words.
sentences = [tokenize(t) for t in X_train]
w2v = Word2Vec(sentences, vector_size=VECTOR_SIZE, window=5, min_count=3,
               sg=1, workers=4, epochs=W2V_EPOCHS, seed=RANDOM_STATE)
kv = w2v.wv
print(f'vocabulary: {len(kv):,} words\n')

# Sanity check worth putting in your report - it shows the embedding learned
# support-domain meaning rather than generic English or punctuation.
for probe in ('refund', 'broken', 'password', 'delayed', 'cancel'):
    if probe in kv:
        print(f'  {probe:<10} -> ' + ', '.join(w for w, _ in kv.most_similar(probe, topn=6)))

In [ ]:
def mean_pool(texts, kv):
    '''Plain average of the word vectors present in the vocabulary.'''
    out = np.zeros((len(texts), kv.vector_size), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs = [kv[w] for w in tokenize(t) if w in kv]
        if vecs:
            out[i] = np.mean(vecs, axis=0)
    return out

def tfidf_weighted_pool(texts, kv, idf_map, default_idf):
    '''
    Weighted average using IDF as the weight.
    In a plain average "the" counts as much as "refund". IDF is small for words
    that appear everywhere and large for rare informative ones, so this pulls the
    ticket vector toward the words that actually carry the intent.
    '''
    out = np.zeros((len(texts), kv.vector_size), dtype=np.float32)
    for i, t in enumerate(texts):
        vecs, weights = [], []
        for w in tokenize(t):
            if w in kv:
                vecs.append(kv[w]); weights.append(idf_map.get(w, default_idf))
        if vecs:
            out[i] = np.average(np.array(vecs), axis=0,
                                weights=np.array(weights, dtype=np.float32))
    return out

tfidf = TfidfVectorizer(min_df=3, max_df=0.6).fit(X_train)
idf_map = dict(zip(tfidf.get_feature_names_out(), tfidf.idf_))
default_idf = float(np.median(tfidf.idf_))
print('IDF weights ready')

In [ ]:
rows, preds = [], {}

def evaluate(name, Xtr, Xte):
    scaler = StandardScaler()
    Xtr, Xte = scaler.fit_transform(Xtr), scaler.transform(Xte)
    models = {
        'LogisticRegression': LogisticRegression(max_iter=3000, class_weight='balanced', n_jobs=-1),
        'LinearSVC': LinearSVC(class_weight='balanced'),
        'RandomForest': RandomForestClassifier(n_estimators=200, min_samples_leaf=2,
                                               n_jobs=-1, class_weight='balanced_subsample',
                                               random_state=RANDOM_STATE),
    }
    print(f'\n--- {name} ---')
    for mname, model in models.items():
        model.fit(Xtr, y_train)
        p = model.predict(Xte)
        f1 = f1_score(y_test, p, average='macro')
        rows.append({'features': name, 'model': mname, 'macro_f1': f1})
        preds[(name, mname)] = p
        print(f'  {mname:<20} macro-F1 {f1:.4f}')

evaluate('W2V mean', mean_pool(X_train, kv), mean_pool(X_test, kv))
evaluate('W2V tfidf-weighted',
         tfidf_weighted_pool(X_train, kv, idf_map, default_idf),
         tfidf_weighted_pool(X_test, kv, idf_map, default_idf))

In [ ]:
if USE_GLOVE:
    import gensim.downloader as api
    print('downloading glove-twitter-200 (once, ~130 MB) ...')
    glove = api.load('glove-twitter-200')
    # GloVe is trained on 2 billion tweets of general English, versus your ~15k
    # support tickets. More general language, less support-specific vocabulary.
    evaluate('GloVe pretrained', mean_pool(X_train, glove), mean_pool(X_test, glove))

In [ ]:
results = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
print(results.to_string(index=False, float_format=lambda v: f'{v:.4f}'))

best = results.iloc[0]
gap = TFIDF_BASELINE - best['macro_f1']
print(f"\nbest embedding : {best['features']} + {best['model']} = {best['macro_f1']:.4f}")
print(f"TF-IDF baseline: {TFIDF_BASELINE:.4f}")
print(f"gap            : {gap:+.4f}")

if gap > 0:
    print('\nTF-IDF wins. For your report: the labels were generated by keyword')
    print('rules, so the signal IS the exact words - which is what TF-IDF encodes')
    print('and what averaging into one vector destroys. Embeddings help when')
    print('meaning matters more than wording; here it does not.')
else:
    print('\nEmbeddings win - say which pooling strategy did it, and why.')

print('\nper-class detail for the best embedding setup:')
print(classification_report(y_test, preds[(best['features'], best['model'])], zero_division=0))

results.to_csv(f'{DRIVE}/ps1_embeddings_comparison.csv', index=False)
print(f'saved -> {DRIVE}/ps1_embeddings_comparison.csv')

In [ ]:
import matplotlib.pyplot as plt

BLUE, PALE, PALER = '#2a78d6', '#9dc3ee', '#c9dcf5'
pivot = results.pivot(index='features', columns='model', values='macro_f1')
fig, ax = plt.subplots(figsize=(8.6, 1.1 * len(pivot) + 2.0))
y = np.arange(len(pivot))
for i, col in enumerate(pivot.columns):
    off = (i - (len(pivot.columns) - 1) / 2) * 0.26
    ax.barh(y + off, pivot[col], height=0.24,
            color=[BLUE, PALE, PALER][i % 3], label=col)
    for yy, v in zip(y + off, pivot[col]):
        ax.text(v + 0.008, yy, f'{v:.3f}', va='center', fontsize=8, color='#0b0b0b')

ax.axvline(TFIDF_BASELINE, color='#b3402f', lw=1.4, ls='--',
           label=f'TF-IDF baseline ({TFIDF_BASELINE:.3f})')
ax.set_yticks(y, pivot.index, fontsize=9)
ax.set_xlim(0, 1.14)
ax.set_xlabel('Macro F1', fontsize=9, color='#52514e')
ax.set_title('PS-1: embedding features vs the TF-IDF baseline',
             fontsize=11, loc='left', pad=12)
ax.legend(frameon=False, fontsize=8.5, ncol=4, loc='upper center',
          bbox_to_anchor=(0.5, -0.18))
ax.tick_params(labelsize=9, colors='#52514e', length=0)
for s in ('top', 'right', 'left'):
    ax.spines[s].set_visible(False)
ax.grid(axis='x', color='#ececea', lw=0.8); ax.set_axisbelow(True)
fig.tight_layout()
fig.savefig(f'{DRIVE}/ps1_embeddings_comparison.png', dpi=160)
plt.show()

---
## For your report

Include the table, the chart, and the nearest-neighbour output.

The nearest neighbours matter more than they look: they show you **inspected** the embedding rather
than trusting it. `refund → refunded, reimburse, refunds` is evidence the model learned support
vocabulary. If instead you saw `refund → refund., refund?, refund!` you would be looking at a
tokenisation bug, not a language model.

Then state which feature set won and why. If TF-IDF wins — likely — the explanation is that your
labels came from keyword rules, so exact wording *is* the signal, and mean-pooling destroys exactly
that. That is a finding about your labelling strategy, not a failure of embeddings.

Both files are saved to Drive: `ps1_embeddings_comparison.csv` and `.png`.